# 01 · Data start (pinned loader)
AI Job-Market Skill Miner (P08). This notebook downloads one fixed Kaggle version of the LinkedIn job postings dataset, so all three members get identical data.

**Run it from a fresh runtime:** Runtime → Disconnect and delete runtime → Run all.

**Before the first run:** add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets (key icon in the left bar) and switch on notebook access. Never paste keys into a cell.

In [ ]:
!pip -q install kagglehub
import os, kagglehub, pandas as pd
from pathlib import Path
from google.colab import userdata

try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    print("Kaggle secrets not set; trying public download")

## 1. Download the pinned version
`DATASET_VERSION` is pinned to **13**, the version recorded in `DATA_MANIFEST.md`. Change it only when the team agrees on a new version and updates the manifest.

In [ ]:
DATASET_VERSION = "13"   # pinned 12 Oct 2026; must match DATA_MANIFEST.md
handle = "arshkon/linkedin-job-postings"
if DATASET_VERSION:
    handle += f"/versions/{DATASET_VERSION}"
folder = Path(kagglehub.dataset_download(handle))
print("Downloaded to:", folder)

for f in sorted(folder.rglob("*.csv")):
    print(f.relative_to(folder), round(f.stat().st_size / 1e6, 1), "MB")

## 2. Load the main postings table

In [ ]:
df = pd.read_csv(folder / "postings.csv")
print("postings shape:", df.shape)
print("columns:", list(df.columns))
display(df.head())

## 3. File table for the manifest
Copy the printed table into the Files section of `DATA_MANIFEST.md`.

In [ ]:
rows = []
for f in sorted(folder.rglob("*.csv")):
    t = df if f.name == "postings.csv" else pd.read_csv(f, low_memory=False)
    rows.append({"file": str(f.relative_to(folder)), "rows": len(t), "columns": t.shape[1]})
print(pd.DataFrame(rows).to_markdown(index=False))

## 4. Numbers for the proposal

In [ ]:
pay_cols = [c for c in ["min_salary", "med_salary", "max_salary"] if c in df.columns]
has_pay = df[pay_cols].notna().any(axis=1)
print(f"Postings: {len(df)} | with any pay: {has_pay.sum()} ({has_pay.mean():.1%})")
if "pay_period" in df.columns:
    print(df.loc[has_pay, "pay_period"].value_counts(dropna=False))
print(df[[c for c in df.columns if "salary" in c]].isna().sum())

## 5. Shared Drive (optional)
Set `MOUNT_DRIVE = True` only when you need the team folder. The shared folder must be added to My Drive as a shortcut first.

In [ ]:
MOUNT_DRIVE = False   # set True when you need the team Drive folder
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = next(Path("/content/drive/MyDrive").glob("AI Job Market Skill Miner*"), None)
    print("Team folder:", root)   # None means the shortcut is missing